## P89-DEVOPS-LIVE-ROLLBACK-REHEARSAL (Lakelink Fiber v4 cutover)
This notebook rehearses the real `ops/v4_cutover_runbook.py` write path (`_swap` / `_rollback_one`) against **live Unity Catalog** objects, as QA S83 criterion 6 recommends. Everything runs in the throwaway scratch schema `cdm_tmforum.pri_scratch_rollback_rehearsal`, which DEC-PRI-P88-OPEN-POINTS-R87 (id 53) item 2 approves.

* **Synthetic fixtures only.** No FROZEN-pin or manifest objects are touched, and nothing is written to `provable_resilience_intelligence`.
* The runbook module is loaded from a /tmp copy, and its `SCHEMA`/`FQ` globals are re-pointed to the scratch schema before any call.
* **Run this interactively.** The scheduled-agent safety filter blocks DEEP CLONE, CREATE OR REPLACE and DROP, even in the scratch schema.
* The last cell drops the scratch schema (DEC-53: drop it afterwards).

**Pass criteria:** every value under `CHECKS` is True.

In [0]:
import sys, os, shutil, importlib, json
REPO = "/Workspace/Users/stephen.hage@databricks.com/provable-resilience-intelligence"
os.makedirs("/tmp/p89rh", exist_ok=True)
shutil.copy(f"{REPO}/ops/v4_cutover_runbook.py", "/tmp/p89rh/rb.py")
sys.path.insert(0, "/tmp/p89rh")
import rb; importlib.reload(rb)
SCRATCH = "pri_scratch_rollback_rehearsal"
rb.SCHEMA, rb.FQ = SCRATCH, f"{rb.CATALOG}.{SCRATCH}"
S = rb.FQ
assert S == "cdm_tmforum.pri_scratch_rollback_rehearsal" and rb.SCHEMA != "provable_resilience_intelligence"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S} COMMENT 'P89 throwaway scratch schema (DEC-53). Synthetic only; drop after run.'")
print(S)

In [0]:
q = spark.sql
q(f"""CREATE OR REPLACE TABLE {S}.rh_span COMMENT 'Lakelink Fiber rehearsal span table (v3 live)' AS
SELECT CAST(id AS INT) span_id, concat('CO-', CAST(id % 7 AS STRING)) a_site,
       CASE WHEN id % 5 = 0 THEN NULL ELSE round(id*1.37,2) END length_km FROM range(0,500)""")
for c, t in [("span_id", "Span identifier"), ("a_site", "A-end central office"), ("length_km", "Span length in km")]:
    q(f"ALTER TABLE {S}.rh_span ALTER COLUMN {c} COMMENT '{t}'")
q(f"""CREATE OR REPLACE TABLE {S}.rh_span_v4_candidate COMMENT 'Lakelink Fiber rehearsal span table (v4 candidate)' AS
SELECT CAST(id AS INT) span_id, concat('CO-', CAST(id % 9 AS STRING)) a_site, round(id*1.41,2) length_km,
       CAST(id % 3 AS INT) srlg_count FROM range(0,620)""")
for c, t in [("span_id", "Span identifier"), ("a_site", "A-end central office"), ("length_km", "Span length in km (v4)"), ("srlg_count", "Shared risk groups on span")]:
    q(f"ALTER TABLE {S}.rh_span_v4_candidate ALTER COLUMN {c} COMMENT '{t}'")
q(f"CREATE OR REPLACE TABLE {S}.rh_newobj_v4_candidate COMMENT 'Lakelink Fiber rehearsal v4-new object' AS SELECT CAST(id AS INT) k FROM range(0,10)")
q(f"ALTER TABLE {S}.rh_newobj_v4_candidate ALTER COLUMN k COMMENT 'key'")
q(f"CREATE OR REPLACE VIEW {S}.rh_view COMMENT 'Lakelink Fiber rehearsal view (v3)' AS SELECT span_id, a_site FROM {S}.rh_span WHERE span_id < 100")
q(f"CREATE OR REPLACE VIEW {S}.rh_view_v4_candidate COMMENT 'Lakelink Fiber rehearsal view (v4)' AS SELECT span_id, a_site, srlg_count FROM {S}.rh_span_v4_candidate WHERE span_id < 200")
q(f"DROP TABLE IF EXISTS {S}.rh_newobj")
q(f"DROP TABLE IF EXISTS {S}.rh_span__v3_rollback")
T = f"{S}.`rh_span`"
snap = [("TABLE", None, 0, "Lakelink Fiber rehearsal span table (v3 live)", f"COMMENT ON TABLE {T} IS 'Lakelink Fiber rehearsal span table (v3 live)';"),
        ("COLUMN", "span_id", 1, "Span identifier", f"ALTER TABLE {T} ALTER COLUMN `span_id` COMMENT 'Span identifier';"),
        ("COLUMN", "a_site", 2, "A-end central office", f"ALTER TABLE {T} ALTER COLUMN `a_site` COMMENT 'A-end central office';"),
        ("COLUMN", "length_km", 3, "Span length in km", f"ALTER TABLE {T} ALTER COLUMN `length_km` COMMENT 'Span length in km';")]
spark.createDataFrame([(rb.V3, "rh_span", *r) for r in snap],
    "manifest_version string, table_name string, comment_level string, column_name string, ordinal_position int, comment string, reapply_sql string"
).write.mode("overwrite").saveAsTable(f"{S}.{rb.V3_COMMENT_SNAPSHOT}")
rows = []
for m, o in [(rb.V3, "rh_span"), (rb.V4_DRAFT, "rh_span_v4_candidate"), (rb.V4_DRAFT, "rh_newobj_v4_candidate")]:
    f = rb.fingerprint(spark, o)
    rows.append((m, o, "PASS", f["fp_row_count"], f["fp_sha2_v1_null_skip"], f["fp_sha2_v2_null_token"], f["fp_column_count"]))
spark.createDataFrame(rows, "manifest_version string, object_name string, gate_status string, fp_row_count bigint, fp_sha2_v1_null_skip string, fp_sha2_v2_null_token string, fp_column_count int"
).write.mode("overwrite").saveAsTable(f"{S}.{rb.FP_GATE_TABLE}")
base = rb._fp_baselines(spark)
print(len(base), "baselines")

In [0]:
R = {}
armed = rb._Armed(spark, None)
def v3state():
    return (rb.compare_fingerprint(rb._safe_fp(spark, "rh_span"), base.get((rb.V3, "rh_span")))["status"],
            rb.comment_coverage_check(spark, "rh_span")["status"])
R["initial"] = v3state()
# A: DEEP CLONE backup -> promote (CREATE OR REPLACE) -> rollback (DEEP CLONE from backup + comment re-apply)
itA = rb.Item("a", "rh_span", "rh_span_v4_candidate")
for s in rb._plan_writes(rb.backup_sql(itA, 1)):
    rb._write(armed, s)
R["A_backup_fp"] = rb.compare_fingerprint(rb._safe_fp(spark, "rh_span__v3_rollback"), base[(rb.V3, "rh_span")])["status"]
fA = rb.Finding(itA, "READY", {"v3_pin": 1, "live_type": "MANAGED"}, rollback_sql=rb.rollback_sql(itA, 1))
R["A_swap"] = rb._swap(armed, fA, base, {})
R["A_rollback"] = rb._rollback_one(armed, fA, base, {})
R["A_after"] = v3state()
# A2: BUG-QA-112 resume case - prior-run PROMOTED object whose Finding carries no rollback_sql
rb._swap(armed, fA, base, {})
fA2 = rb.Finding(itA, "V3_FP_MISMATCH", {"v3_pin": 1, "live_type": "MANAGED"}, rollback_sql=[])
R["A2_rollback"] = rb._rollback_one(armed, fA2, base, {})
R["A2_after"] = v3state()
# B: view repoint -> rollback from captured pre-swap DDL (PRE_VIEW_DDL path)
itB = rb.Item("b", "rh_view", "rh_view_v4_candidate", action="repoint_view")
fB = rb.Finding(itB, "READY", {"live_type": "VIEW"})
pre = rb._capture_view_ddl(spark, fB); vd = {"rh_view": pre}
R["B_swap"] = rb._swap(armed, fB, base, vd)
R["B_cols_repointed"] = rb._columns(spark, "rh_view")
R["B_rollback"] = rb._rollback_one(armed, fB, base, vd)
R["B_ddl_equal"] = rb._show_create(spark, "rh_view").strip() == pre.strip()
# C: v4-new object (no v3 pin) -> rollback is DROP
itC = rb.Item("c", "rh_newobj", "rh_newobj_v4_candidate")
fC = rb.Finding(itC, "READY", {"v3_pin": None}, rollback_sql=rb.rollback_sql(itC, None))
# INFO: CTAS drops the table COMMENT and a v4-new object has no v3 snapshot rows, so _swap may fail closed with
# POST_SWAP_COMMENTS_TABLE_MISSING (swapped=True). That is recorded for @qa, and the DROP rollback must still work.
try:
    R["C_swap"] = rb._swap(armed, fC, base, {})
except rb._StepFailed as e:
    R["C_swap"] = ("STEP_FAILED", e.reason, e.swapped)
R["C_rollback"] = rb._rollback_one(armed, fC, base, {})
R["C_gone"] = "rh_newobj" not in rb._objects(spark)
# D: corrupted backup -> swap must refuse BEFORE CREATE OR REPLACE; live stays v3
spark.sql(f"INSERT INTO {S}.rh_span__v3_rollback VALUES (9999, 'CO-X', 1.0)")
try:
    rb._swap(armed, fA, base, {}); R["D"] = "NOT_REFUSED"
except rb._StepFailed as e:
    R["D"] = (e.reason, e.swapped)
R["D_after"] = v3state()
CHECKS = {
    "backup_fp_pass": R["A_backup_fp"] == "FP_PASS",
    "A_swap_fp_pass": R["A_swap"]["fp"] == "FP_PASS" and R["A_swap"]["comments"] == rb.COMMENTS_OK,
    "A_rolled_back": R["A_rollback"] == "ROLLED_BACK" and R["A_after"] == ("FP_PASS", rb.COMMENTS_OK),
    "A2_rolled_back_empty_plan": R["A2_rollback"] == "ROLLED_BACK" and R["A2_after"] == ("FP_PASS", rb.COMMENTS_OK),
    "B_repointed": "srlg_count" in R["B_cols_repointed"],
    "B_restored": R["B_rollback"] == "ROLLED_BACK" and R["B_ddl_equal"],
    "C_dropped": R["C_rollback"] == "ROLLED_BACK" and R["C_gone"],
    "D_refused_unswapped": isinstance(R["D"], tuple) and R["D"][0].startswith("BACKUP_") and R["D"][1] is False and R["D_after"] == ("FP_PASS", rb.COMMENTS_OK),
}
print(json.dumps(R, indent=1, default=str)); print("CHECKS", CHECKS, "ALL_PASS", all(CHECKS.values()), len(armed.statements), "writes")

In [0]:
assert S == "cdm_tmforum.pri_scratch_rollback_rehearsal"
spark.sql(f"DROP SCHEMA IF EXISTS {S} CASCADE")
print("dropped", S)